In [1]:
from langgraph.graph import StateGraph, START , END
from typing import TypedDict

In [2]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes:int

    sr:float
    bpb:float
    boundary_precentage:float
    summary :str

In [17]:
def calculate_sr(state: BatsmanState):
    sr=(state['runs']/state['balls'])*100
    return {'sr':sr}

In [18]:
def calculate_bpb(state:BatsmanState):
    bpb=state['balls']/(state['fours']+state['sixes'])
    return {'bpb':bpb}

In [19]:
def calculate_boundary_percent(state:BatsmanState)->BatsmanState:
    boundary_percent=(((state['fours']*4)+(state['sixes']*6))/state['runs'])*100
    return {'boundary_precentage':boundary_percent}

In [56]:
def summary(state:BatsmanState)->BatsmanState:
    summary = (
        f"Strike rate - {state['sr']}\n"
        f"Balls per boundary - {state['bpb']}\n"
        f"Boundary percent - {state['boundary_precentage']}"
    )
    return {'summary':summary}

In [58]:
graph=StateGraph(BatsmanState)

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percent',calculate_boundary_percent)
graph.add_node('summary',summary)


#edges
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge(START,'calculate_bpb')


graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percent','summary')

graph.add_edge('summary',END)




In [59]:
workflow=graph.compile()

In [49]:
from langchain_core.runnables.graph import MermaidDrawMethod


graph.compile().get_graph().draw_mermaid()

'---\nconfig:\n  flowchart:\n    curve: linear\n---\ngraph TD;\n\t__start__([<p>__start__</p>]):::first\n\tcalculate_sr(calculate_sr)\n\tcalculate_bpb(calculate_bpb)\n\tcalculate_boundary_percent(calculate_boundary_percent)\n\tsummary(summary)\n\t__end__([<p>__end__</p>]):::last\n\t__start__ --> calculate_boundary_percent;\n\t__start__ --> calculate_bpb;\n\t__start__ --> calculate_sr;\n\tcalculate_boundary_percent --> summary;\n\tcalculate_bpb --> summary;\n\tcalculate_sr --> summary;\n\tsummary --> __end__;\n\tclassDef default fill:#f2f0ff,line-height:1.2\n\tclassDef first fill-opacity:0\n\tclassDef last fill:#bfb6fc\n'

In [40]:
import requests


resp=requests.post(
    "https://mermaid.ink/img",
    headers={"User-Agent": "Mozilla/5.0"},
    data={"code": "graph TD; A-->B;"}
)

print(resp.status_code)
print(resp)

404
<Response [404]>


In [60]:
initial_state={
    'runs':100,
    'balls':50,
    'fours':4,
    'sixes':5
}

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 50,
 'fours': 4,
 'sixes': 5,
 'sr': 200.0,
 'bpb': 5.555555555555555,
 'boundary_precentage': 46.0,
 'summary': 'Strike rate - 200.0\nBalls per boundary - 5.555555555555555\nBoundary percent - 46.0'}